In [9]:
from langchain_openai import ChatOpenAI
from langchain_groq import ChatGroq
from typing import Annotated, TypedDict, Optional, Literal, List
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage
from pydantic import BaseModel, Field

load_dotenv()

True

In [10]:
menu_items ="""
Appetizers
----------
1. Garlic Bread - Toasted baguette slices brushed with garlic butter and herbs. - $5.99
2. Chicken Wings - Crispy fried wings tossed in spicy buffalo sauce. - $8.49
3. Paneer Tikka - Marinated paneer cubes grilled with bell peppers and onions. - $7.99

Main Course
-----------
4. Margherita Pizza - Classic pizza with fresh mozzarella, tomato sauce, and basil. - $12.99
5. Butter Chicken - Tender chicken simmered in a creamy tomato and butter gravy. - $14.49
6. Veg Biryani - Fragrant basmati rice cooked with mixed vegetables and aromatic spices. - $11.99
7. Grilled Salmon - Atlantic salmon fillet served with lemon butter sauce and steamed vegetables. - $17.99

Desserts
--------
8. Chocolate Lava Cake - Warm chocolate cake with a molten center, served with vanilla ice cream. - $6.49
9. Gulab Jamun - Soft milk dumplings soaked in rose-flavored sugar syrup. - $4.99

Beverages
---------
10. Mango Lassi - Chilled yogurt drink blended with ripe mangoes and a hint of cardamom. - $3.99

 """

menu_items

'\nAppetizers\n----------\n1. Garlic Bread - Toasted baguette slices brushed with garlic butter and herbs. - $5.99\n2. Chicken Wings - Crispy fried wings tossed in spicy buffalo sauce. - $8.49\n3. Paneer Tikka - Marinated paneer cubes grilled with bell peppers and onions. - $7.99\n\nMain Course\n-----------\n4. Margherita Pizza - Classic pizza with fresh mozzarella, tomato sauce, and basil. - $12.99\n5. Butter Chicken - Tender chicken simmered in a creamy tomato and butter gravy. - $14.49\n6. Veg Biryani - Fragrant basmati rice cooked with mixed vegetables and aromatic spices. - $11.99\n7. Grilled Salmon - Atlantic salmon fillet served with lemon butter sauce and steamed vegetables. - $17.99\n\nDesserts\n--------\n8. Chocolate Lava Cake - Warm chocolate cake with a molten center, served with vanilla ice cream. - $6.49\n9. Gulab Jamun - Soft milk dumplings soaked in rose-flavored sugar syrup. - $4.99\n\nBeverages\n---------\n10. Mango Lassi - Chilled yogurt drink blended with ripe mango

In [11]:
class MenuItem(BaseModel):
    name: str = Field(..., description="The name of the menu item")
    description: str = Field(..., description="The description of the menu item")
    price: float = Field(..., description="The price of the menu item")
    category: Optional[str] = Field(None, description="Category: Appetizers, Main Course, Desserts, or Beverages")


class MenuResponse(BaseModel):
    reply: str = Field(..., description="A short conversational reply to the user")
    items: List[MenuItem] = Field(default_factory=list, description="Menu items relevant to the user's question. Empty if the question is not about specific items.")

In [ ]:
template = PromptTemplate(
    input_variables=["menu_items"],
    template="You are a helpful assistant that provides information about the menu items. Here is the menu:\n\n{menu_items}\n\nPlease provide details about the menu items when asked.")

prompt=template.invoke({"menu_items": menu_items})

print(prompt)

text='You are a helpful assistant that provides information about the menu items. Here is the menu:\n\n\nAppetizers\n----------\n1. Garlic Bread - Toasted baguette slices brushed with garlic butter and herbs. - $5.99\n2. Chicken Wings - Crispy fried wings tossed in spicy buffalo sauce. - $8.49\n3. Paneer Tikka - Marinated paneer cubes grilled with bell peppers and onions. - $7.99\n\nMain Course\n-----------\n4. Margherita Pizza - Classic pizza with fresh mozzarella, tomato sauce, and basil. - $12.99\n5. Butter Chicken - Tender chicken simmered in a creamy tomato and butter gravy. - $14.49\n6. Veg Biryani - Fragrant basmati rice cooked with mixed vegetables and aromatic spices. - $11.99\n7. Grilled Salmon - Atlantic salmon fillet served with lemon butter sauce and steamed vegetables. - $17.99\n\nDesserts\n--------\n8. Chocolate Lava Cake - Warm chocolate cake with a molten center, served with vanilla ice cream. - $6.49\n9. Gulab Jamun - Soft milk dumplings soaked in rose-flavored sugar 

In [12]:
chat_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a helpful restaurant assistant. Answer questions about the menu below."
     "Always respond using the structured schema: a short conversational `reply`, "
     "and `items` listing any menu items the user asked about (empty list if none)."
     "MENU: {menu_items}"),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}"),
])

In [13]:
model = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)
structured_model = model.with_structured_output(MenuResponse)
chain = chat_prompt | structured_model

history = []

def ask(user_input: str):
    response = chain.invoke({
        "menu_items": menu_items,
        "history": history,
        "input": user_input,
    })
    # Append turn to history so the next call has context
    history.append(HumanMessage(content=user_input))
    history.append(AIMessage(content=response.reply))
    return response

# Try it out
r1 = ask("What appetizers do you have?")
print(r1.reply)
for it in r1.items:
    print(f"  - {it.name} (${it.price})")

print()
r2 = ask("Which of those is the cheapest?")
print(r2.reply)
for it in r2.items:
    print(f"  - {it.name} (${it.price})")

e:\workspace\langchain-mastery\.venv\Lib\site-packages\langchain_openai\chat_models\base.py:2381: UserWarning: Cannot use method='json_schema' with model gpt-3.5-turbo since it doesn't support OpenAI's Structured Output API. You can see supported models here: https://platform.openai.com/docs/guides/structured-outputs#supported-models. To fix this warning, set `method='function_calling'. Overriding to method='function_calling'.
  warnings.warn(


We have the following appetizers available: Garlic Bread, Chicken Wings, and Paneer Tikka.
  - Garlic Bread ($5.99)
  - Chicken Wings ($8.49)
  - Paneer Tikka ($7.99)

The cheapest appetizer we have is the Garlic Bread.
  - Garlic Bread ($5.99)
